In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib_etl.s3 import etl_input_data_validator
from lib.s3 import etl_input_table_validator
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Transform 

In [0]:
recency_lookback_duration = data_paths.get("recency_lookback_duration", {})
etl_input_table_validator(
    silver_master_email, silver_fiscal_days, 
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)

In [0]:
email = spark.table(silver_master_email)
email.createOrReplaceTempView("email")

fiscal_days = spark.table(
    silver_fiscal_days
)
fiscal_days.createOrReplaceTempView("fiscal_days")


df_email_fiscal = spark.sql("""
SELECT e.*, f.FISCAL_WEEK_START, f.FISCAL_WEEK_END
FROM email e
JOIN fiscal_days f
ON e.FIRST_OPEN_DATE = f.FISCAL_DAY
""")

df_email_fiscal.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'email_fiscal', config_validation, df_email_fiscal, stats_etl_path
    )

### Merge

In [0]:
df_email_fiscal.write.mode("overwrite").saveAsTable(silver_master_email_fiscal)

if archive_flag:
    save_archive(df_email_fiscal, silver_master_email_fiscal_archive, run_as_date)